# Polynomial Features - Regression Companion Notebook

This notebook is the practical code companion for the regression part of [`lecture_notes/13_polynomial_features.pdf`](../../lecture_notes/13_polynomial_features.pdf). The focus here is on making polynomial feature expansion concrete: what new columns are created, how they change the design matrix, and how they allow a linear model to fit curved relationships.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

plt.style.use("seaborn-v0_8-whitegrid")
SEED = 42
rng = np.random.default_rng(SEED)


This setup cell imports the tools used throughout the notebook. The experiments stay deliberately simple so that the main object of interest is the feature expansion itself rather than the optimization algorithm.


## Inspect a Polynomial Feature Expansion

We start with a tiny matrix so that every generated feature can be inspected directly.


In [ ]:
X_small = np.arange(6).reshape(3, 2)
X_small


In [ ]:
poly2 = PolynomialFeatures(degree=2, include_bias=False)
X_small_poly2 = poly2.fit_transform(X_small)

pd.DataFrame(
    X_small_poly2,
    columns=poly2.get_feature_names_out(["x1", "x2"]),
)


This table makes the feature expansion explicit. The model will still be linear in its parameters, but it now operates on a richer design matrix containing squares and interaction terms such as `x1 x2`.


In [ ]:
poly3 = PolynomialFeatures(degree=3, include_bias=False)
X_small_poly3 = poly3.fit_transform(X_small)

print("original shape:", X_small.shape)
print("degree-2 shape:", X_small_poly2.shape)
print("degree-3 shape:", X_small_poly3.shape)
print("degree-3 columns:", poly3.get_feature_names_out(["x1", "x2"]))


Increasing the degree enlarges the feature space quickly. This is one of the main practical consequences of polynomial expansion: the model becomes more expressive, but the representation also becomes larger and more prone to overfitting.


## From Linear Regression to Polynomial Regression

We now move to a one-dimensional regression problem with a curved underlying relationship.


In [ ]:
def f(x):
    return np.sin(1.5 * np.pi * x)


X = np.linspace(0, 1, 30)
y = f(X) + rng.normal(0, 0.15, size=X.shape[0])

X = X.reshape(-1, 1)
X_plot = np.linspace(0, 1, 300).reshape(-1, 1)
y_true = f(X_plot.ravel())


In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(X, y, color="tab:blue", label="training data")
plt.plot(X_plot, y_true, color="black", label="true function")
plt.xlabel("x")
plt.ylabel("y")
plt.title("A curved regression problem")
plt.legend()
plt.show()


This plot motivates the feature expansion: a straight line is too restrictive for this problem. The goal is not to change the learning algorithm, but to change the representation seen by the linear model.


In [ ]:
degrees = [1, 2, 3, 5]
models = {}
for degree in degrees:
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("linreg", LinearRegression()),
    ])
    model.fit(X, y)
    models[degree] = model


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
axes = axes.ravel()

for ax, degree in zip(axes, degrees):
    y_hat = models[degree].predict(X_plot)
    ax.scatter(X, y, color="tab:blue", alpha=0.8)
    ax.plot(X_plot, y_true, color="black", label="true function")
    ax.plot(X_plot, y_hat, color="tab:red", label=f"degree {degree}")
    ax.set_title(f"Polynomial degree {degree}")
    ax.grid(True)

axes[0].legend()
plt.tight_layout()
plt.show()


This figure is the main regression result. Degree 1 corresponds to ordinary linear regression, while larger degrees allow the fitted curve to bend. Reading the panels side by side makes the transition from underfitting to more flexible fits easy to see.


In [ ]:
rows = []
for degree, model in models.items():
    poly = model.named_steps["poly"]
    rows.append({
        "degree": degree,
        "n_generated_features": poly.fit_transform(X[:1]).shape[1],
    })

pd.DataFrame(rows)


Even in one dimension, the number of derived columns grows with the degree. In higher-dimensional problems, this growth becomes much faster, which is why the later notes on overfitting, regularization, and model selection become so important.


## A Two-Feature Interaction Example

Polynomial features are not only about powers such as `x^2`; they also include interactions between inputs.


In [ ]:
X_interaction = np.array([
    [2, 3],
    [4, 1],
    [5, 2],
])

poly_inter = PolynomialFeatures(degree=2, include_bias=False)
pd.DataFrame(
    poly_inter.fit_transform(X_interaction),
    columns=poly_inter.get_feature_names_out(["x1", "x2"]),
)


The interaction term `x1 x2` is often the most conceptually important one. It lets the model represent relationships in which the effect of one variable depends on the value of another variable.
